# MCP Greek Search and CTS Navigation

This notebook uses the `perseus` MCP server to search and navigate real Greek data. It demonstrates both Unicode Greek and Beta Code input for Scaife search, then uses CTS navigation tools to inspect neighboring URNs.

> Requirements: run from the repository root (or keep the path setup cell unchanged), install dependencies, and have internet access to Perseus/Scaife upstream services.


In [1]:
from pathlib import Path
import importlib
import json
import sys
import xml.etree.ElementTree as ET

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from fastmcp import Client
import server

# Reload local edits when this notebook is rerun in an existing kernel.
server = importlib.reload(server)
mcp = server.mcp
_normalize_greek_query = server._normalize_greek_query


def tool_text(result):
    return "\n".join(
        block.text for block in result.content if getattr(block, "text", None) is not None
    )


## Confirm Beta Code normalization

The MCP server accepts Unicode Greek directly, but it can also normalize common Beta Code before sending the query to Scaife. This is useful if your notes or older datasets encode polytonic Greek as ASCII.


In [2]:
for query in ["μῆνιν", "mh=nin", "a)/eide qea/"]:
    print(f"{query!r} -> {_normalize_greek_query(query)}")


'μῆνιν' -> μῆνιν
'mh=nin' -> μῆνιν
'a)/eide qea/' -> ἄειδε θεά


## Search real Greek text via the MCP tool

`search_perseus` calls Scaife's search API through the MCP server. The first call searches for the opening word of the *Iliad* as Unicode Greek; the second searches a two-word phrase as Beta Code.


In [3]:
async with Client(mcp) as client:
    unicode_search = await client.call_tool(
        "search_perseus",
        {"query": "μῆνιν", "language": "greek", "query_format": "unicode"},
    )
    betacode_search = await client.call_tool(
        "search_perseus",
        {"query": "a)/eide qea/", "language": "greek", "query_format": "betacode"},
    )

unicode_results = json.loads(tool_text(unicode_search))
betacode_results = json.loads(tool_text(betacode_search))

print(json.dumps(unicode_results, ensure_ascii=False, indent=2)[:3000])
print("\n--- Beta Code query results ---")
print(json.dumps(betacode_results, ensure_ascii=False, indent=2)[:3000])


{
  "results": [
    {
      "passage": {
        "url": "/reader/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/",
        "json_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/json/",
        "text_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/text/",
        "text": {
          "url": "/library/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/",
          "json_url": "/library/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/json/",
          "text_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/text/",
          "ancestors": [
            {
              "url": "/library/urn:cts:greekLit:tlg2045/",
              "json_url": "/library/urn:cts:greekLit:tlg2045/json/",
              "text_url": "/library/passage/urn:cts:greekLit:tlg2045/text/",
              "urn": "urn:cts:greekLit:tlg2045",
              "label": "Nonnus of Panopolis"
            },
            {
              "url": "/library/urn:cts:greekL

## Get valid references for the *Iliad*

CTS references are hierarchical. `get_valid_references` can list valid citation values for a work or edition. The example below passes `level=1` for the Greek *Iliad* edition. The live Perseus CTS endpoint may still return passage-level references, so inspect the response rather than assuming a specific citation depth.


In [4]:
HOMER_TEXTGROUP = "urn:cts:greekLit:tlg0012"

async with Client(mcp) as client:
    homer = await client.call_tool(
        "get_author_resources",
        {"author": HOMER_TEXTGROUP, "language": "greek"},
    )

homer_json = json.loads(tool_text(homer))
if not homer_json["authors"]:
    raise RuntimeError("No Homer resources were returned by the CTS inventory.")

iliad_work = next(
    (work for work in homer_json["authors"][0]["works"] if "Iliad" in work["titles"]),
    None,
)
if iliad_work is None or not iliad_work["editions"]:
    raise RuntimeError("No Greek Iliad edition is advertised by the CTS inventory.")

ILIAD_EDITION = iliad_work["editions"][0]["urn"]

async with Client(mcp) as client:
    refs = await client.call_tool(
        "get_valid_references",
        {"urn": ILIAD_EDITION, "level": 1},
    )

refs_xml = tool_text(refs)
print(refs_xml[:3000])


<GetValidReff xmlns="http://chs.harvard.edu/xmlns/cts3/ti">
  <request>
    <requestName>GetValidReff</requestName>
    <requestUrn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1</requestUrn>
  </request>
  <reply>
    <reff>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.1</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.2</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.3</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.4</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.5</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.6</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.7</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.8</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.9</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.11</urn>
      <urn>urn:cts:greekLit:tlg0012.tlg001.perse

## Navigate around a specific Greek passage

`get_prev_next_urn` first asks CTS for neighboring citations. If the live CTS endpoint returns malformed HTML instead of the expected XML, the server derives a well-formed response from ordered `GetValidReff` results. This is useful in a reading interface or research workflow where an agent needs to move from one passage to the next without guessing URN syntax.


In [5]:
CURRENT = f"{ILIAD_EDITION}:1.10"

async with Client(mcp) as client:
    nav = await client.call_tool("get_prev_next_urn", {"urn": CURRENT})
    current_text = await client.call_tool("get_passage_plaintext", {"urn": CURRENT})

print(tool_text(nav))
print("\nCurrent passage text:")
print(tool_text(current_text))


<GetPrevNextUrn><request><requestName>GetPrevNextUrn</requestName><requestUrn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10</requestUrn></request><reply><previous><urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.9</urn></previous><next><urn>urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.11</urn></next></reply></GetPrevNextUrn>

Current passage text:
νοῦσον ἀνὰ στρατὸν ὄρσε κακήν, ὀλέκοντο δὲ λαοί,


## Optional: parse navigation XML

The CTS navigation response is XML. The exact namespace details can vary, so this lightweight parser prints elements that contain URN-like text or attributes. It is intentionally exploratory.


In [6]:
root = ET.fromstring(tool_text(nav))
for element in root.iter():
    local_name = element.tag.rsplit("}", 1)[-1]
    values = []
    if element.text and "urn:cts:" in element.text:
        values.append(element.text.strip())
    for value in element.attrib.values():
        if "urn:cts:" in value:
            values.append(value)
    if values:
        print(local_name, "->", values)


requestUrn -> ['urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10']
urn -> ['urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.9']
urn -> ['urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.11']


## What this shows

These cells show the MCP connection doing four research tasks over real Greek data:

- normalizing Greek search input;
- searching Perseus/Scaife;
- discovering valid CTS references;
- navigating and fetching passage text by URN.

The same tool names and arguments are the ones exposed to MCP-capable applications such as Cursor, Claude Desktop, or the MCP Inspector.


# Notebook version

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.0</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 3, 2026</td>
    </tr>
  </table>
</div>